# Quantum Collapse and Revival — Hamiltonian Time Evolution (TEBD)

Demonstrates the `/evolve` endpoints: TEBD (Time-Evolving Block Decimation) Suzuki-Trotter evolution of quantum spin chains, with emphasis on two physically striking phenomena:

1. **Collapse and revival** — a quantum quench drives the system to a "collapsed" state; coherence revives at the Poincaré recurrence time
2. **Kibble-Zurek mechanism (QKZM)** — slow ramps through a quantum phase transition produce topological defects at a rate predicted by the KZM scaling law

**Engine used:** `KLTMPSState` with TEBD bond unitaries, up to bond dimension χ = 128.

**What the `/evolve` API returns:**
- `trajectory`: time-resolved observables (entropy, magnetisation, QFI, ZZ correlators)
- `kzm_defect_density` and `kzm_prediction`: QKZM scaling signature
- `C_tR`: collapse-revival correlator heatmap C(t, R)

In [ ]:
%pip install qumulator-sdk matplotlib --quiet
import os, sys
API_KEY = os.environ.get('QUMULATOR_API_KEY', 'YOUR_KEY_HERE')
API_URL = 'https://api.qumulator.com'
print('Python:', sys.version.split()[0])

In [ ]:
from qumulator import QumulatorClient
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

client = QumulatorClient(api_url=API_URL, api_key=API_KEY)

## 1. Real-time evolution — Transverse-Field Ising Model

We evolve a 10-site TFIM (J=1, h=1, critical point) starting from the ferromagnetic ground state |↑↑↑...⟩. Observables recorded every dt = 0.1 up to t = 2.0.

In [ ]:
evolve_result = client.evolve.run(
    n_qubits=10,
    hamiltonian={"preset": "ising_1d", "J": 1.0, "h": 1.0},
    t_max=2.0,
    dt=0.1,
    bond_dim=64,
    observables=["entropy", "magnetization", "qfi"],
    initial_state="ferromagnet",
    order=2,
)

traj = evolve_result.trajectory
t_vals  = [pt["t"]                                      for pt in traj]
entropy = [pt.get("max_entropy", pt.get("entropy", 0))  for pt in traj]
mag     = [pt.get("magnetization", pt.get("mag", 0))    for pt in traj]
qfi     = [pt.get("f_Q_density",  pt.get("qfi", 0))    for pt in traj]

print(f"Evolution complete: {len(traj)} trajectory points")
print(f"\n{'t':>6}  {'S_max':>8}  {'<M>':>8}  {'QFI':>8}")
print("-" * 36)
for i in range(0, len(traj), 4):
    pt = traj[i]
    print(f"  {t_vals[i]:4.1f}  {entropy[i]:8.4f}  {mag[i]:8.4f}  {qfi[i]:8.4f}")

In [ ]:
# Plot entropy and magnetisation vs time
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(t_vals, entropy, 'b-o', ms=4)
axes[0].set_xlabel('t')
axes[0].set_ylabel('S_max (von Neumann)')
axes[0].set_title('Entanglement entropy builds up\nafter quench from ferromagnet')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_vals, mag, 'r-o', ms=4)
axes[1].set_xlabel('t')
axes[1].set_ylabel('<M>')
axes[1].set_title('Magnetisation collapses\nunder transverse field')
axes[1].axhline(0, ls='--', c='k', alpha=0.5)
axes[1].grid(True, alpha=0.3)

axes[2].plot(t_vals, qfi, 'g-o', ms=4)
axes[2].set_xlabel('t')
axes[2].set_ylabel('f_Q density')
axes[2].set_title('QFI builds up:\nf_Q > k  certifies (k+1)-partite entanglement')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('tebd_observables.png', dpi=120, bbox_inches='tight')
plt.show()
print("Fig saved: tebd_observables.png")

## 2. Collapse-and-revival quench

A sudden quench: start from the ferromagnetic ground state, suddenly switch on a large transverse field h >> J, evolve. The magnetisation collapses and revives periodically. This is a quantum analogue of Poincaré recurrence.

The correlator C(t, R) = ⟨σ^z_0 σ^z_R⟩(t) encodes both the space and time structure of the revival.

In [ ]:
quench_result = client.evolve.quench(
    n_qubits=20,
    h=3.0,      # large transverse field: collapse regime
    t_max=8.0,
    bond_dim=64,
    observables=["magnetization", "correlator"],
)

qtraj = quench_result.trajectory
qt_vals = [pt["t"] for pt in qtraj]
qmag    = [pt.get("magnetization", 0) for pt in qtraj]

print(f"Quench trajectory: {len(qtraj)} points  (t=0 to {qt_vals[-1]:.2f})")

# Find first revival: when |<M>| crosses 0.5 again after initial collapse
crossed = [(i, t) for i, (t, m) in enumerate(zip(qt_vals, qmag)) if abs(m) > 0.5 and t > 1.0]
t_revival = crossed[0][1] if crossed else float('nan')
print(f"First revival at t ≈ {t_revival:.2f}  (expected t_rev = π/h ≈ {np.pi/3.0:.2f})")
print()

# Print trajectory snapshot
print(f"{'t':>6}  {'<M>':>10}")
print("-" * 20)
for i in range(0, min(len(qtraj), 20), 2):
    print(f"  {qt_vals[i]:4.1f}  {qmag[i]:10.4f}")

In [ ]:
# C(t, R) heatmap if available in response
C_tR = getattr(quench_result, 'C_tR', None) or {}
if C_tR and isinstance(C_tR, dict) and 'data' in C_tR:
    data = np.array(C_tR['data'])
    t_grid = C_tR.get('t_vals', qt_vals)
    R_max  = data.shape[1] if data.ndim == 2 else 0
    if R_max > 0:
        fig, ax = plt.subplots(figsize=(10, 4))
        im = ax.imshow(data.T, aspect='auto', origin='lower',
                       extent=[t_grid[0], t_grid[-1], 0, R_max],
                       cmap='RdBu_r', vmin=-1, vmax=1)
        plt.colorbar(im, ax=ax, label='C(t, R)')
        ax.set_xlabel('t')
        ax.set_ylabel('R (distance from origin)')
        ax.set_title('ZZ Correlator C(t, R) = ⟨σ⁰ᶻ σᴿᶻ⟩(t)\nCollapse-and-revival quench')
        plt.tight_layout()
        plt.savefig('C_tR_heatmap.png', dpi=120, bbox_inches='tight')
        plt.show()
        print("C(t,R) heatmap saved: C_tR_heatmap.png")
    else:
        print("C(t,R) data not returned for this request (observables kwarg may be needed).")
else:
    # Plot magnetisation instead as a proxy for the revival
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(qt_vals, qmag, 'b-', lw=1.5)
    ax.set_xlabel('t')
    ax.set_ylabel('<M>')
    ax.set_title('Magnetisation — collapse and revival after sudden quench (h=3)')
    ax.axhline(0, ls='--', c='k', alpha=0.3)
    plt.tight_layout()
    plt.savefig('quench_revival.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Revival plot saved: quench_revival.png")

## 3. QKZM — Kibble-Zurek Quench Protocol

The Kibble-Zurek mechanism (KZM) predicts that when a system is ramped through a quantum phase transition at a finite rate, topological defects are produced at a universal rate $n_d \propto \tau_Q^{-1/2}$ where $\tau_Q$ is the ramp time.

The Qumulator `/evolve/qkzm` endpoint implements this protocol: ramp $h$ from a disordered phase ($h_0 \gg J$) to an ordered phase ($h_f < J$) at a controlled rate, then measure the defect density.

In [ ]:
qkzm_result = client.evolve.qkzm(
    n_qubits=20,
    J=1.0,
    h0=5.0,    # start in disordered phase (h >> J)
    h_f=0.2,   # end in ordered phase (h << J)
    t_ramp=5.0,
    bond_dim=64,
)

defect_density = qkzm_result.kzm_defect_density
kzm_prediction = qkzm_result.kzm_prediction
tau_Q          = getattr(qkzm_result, 'tau_Q', None)

print("QKZM Protocol Results:")
print(f"  Measured defect density  n_d  = {defect_density:.4f}")
print(f"  KZM predicted n_d             = {kzm_prediction:.4f}  (n_d ~ tau_Q^{{-1/2}})")
if tau_Q:
    print(f"  tau_Q                         = {tau_Q:.4f}")
    print(f"  tau_Q^{{-1/2}} (KZM scaling)    = {tau_Q**(-0.5):.4f}")

agreement = abs(defect_density - kzm_prediction) / (kzm_prediction + 1e-10)
print(f"\n  Agreement with KZM prediction: {100*(1 - agreement):.1f}%")
print()
print("→ The KZM universality class is σ = ν·z = 1 for 1D TFIM.")
print("  Defect density scales as n_d ∝ (dh/dt)^{1/2}.")

## 4. Imaginary-time ground state

Evolve in imaginary time τ → -iτ to project onto the ground state. The energy converges to E₀ monotonically from above.

In [ ]:
gs_result = client.evolve.ground(
    n_qubits=10,
    hamiltonian={"preset": "ising_1d", "J": 1.0, "h": 1.0},
    bond_dim=64,
    n_steps=300,
)

energy     = gs_result.energy
bond_entropy = gs_result.bond_entropy
converged  = getattr(gs_result, 'converged', True)

# Exact ground state energy for N=10 TFIM (J=h=1): from exact diagonalisation
# E_0/N ≈ -1.2381 for N=10
E_exact_per_site = -1.2381
E_TEBD_per_site  = energy / 10

print("Imaginary-time ground state preparation:")
print(f"  E₀ (TEBD)          = {energy:.6f}  ({E_TEBD_per_site:.6f} per site)")
print(f"  E₀ (exact, N=10)   ≈ {E_exact_per_site * 10:.6f}  ({E_exact_per_site:.6f} per site)")
print(f"  Error              = {abs(E_TEBD_per_site - E_exact_per_site):.6f} per site  ({100*abs(E_TEBD_per_site - E_exact_per_site)/abs(E_exact_per_site):.3f}%)")
print(f"  Converged          = {converged}")
print()
print("Bond entropy profile (von Neumann per bond):")
if bond_entropy:
    for i, s in enumerate(bond_entropy):
        bar = '█' * int(s * 20)
        print(f"  Bond {i}-{i+1}: {s:.4f}  {bar}")

## Summary

| Endpoint | What it does | Key output |
|---|---|---|
| `client.evolve.run()` | TEBD real-time evolution (Suzuki-Trotter) | `trajectory` of entropy, magnetization, QFI per timestep |
| `client.evolve.quench()` | Sudden quench, collapse-and-revival | `trajectory` + `C_tR` correlator heatmap |
| `client.evolve.qkzm()` | KZM quench protocol | `kzm_defect_density`, `kzm_prediction`, `tau_Q` |
| `client.evolve.ground()` | Imaginary-time ground state | `energy`, `bond_entropy`, `converged` |
| `client.evolve.lattice()` | 2D lattice regime classifier | `bond_entropy_2d` heatmap, `phase_label` |

**QFI interpretation (Tóth–Gühne 2012):** `f_Q > k` certifies genuine $(k+1)$-partite entanglement. The TEBD trajectory shows how multi-partite entanglement builds up from a product state.